# Phase 0: Mathematical & Computational Foundations  
## Notebook 00_03 — Numerical Linear Algebra for Finance

### Research question

Why is numerical linear algebra essential in quantitative finance, and how do matrix operations such as covariance estimation, eigenvalue decomposition, Cholesky factorisation, PCA, and matrix inversion affect portfolio risk?

This notebook introduces the core linear algebra tools that appear repeatedly in later notebooks:

1. **Return vectors and covariance matrices**
2. **Positive semi-definiteness and eigenvalues**
3. **Cholesky factorisation for correlated simulations**
4. **Principal Component Analysis for risk-factor decomposition**
5. **Condition numbers and instability in matrix inversion**
6. **Covariance shrinkage as a practical stabilisation technique**

The goal is to build intuition for why portfolio optimisation, risk modelling, Monte Carlo simulation, factor models, and volatility surface methods all depend on stable numerical linear algebra.

## 1. Theory: vectors, covariance, and quadratic forms

Suppose we have $N$ assets and a vector of returns:

$$
r_t =
\begin{bmatrix}
r_{t,1} \\
r_{t,2} \\
\vdots \\
r_{t,N}
\end{bmatrix}
$$

The covariance matrix is:

$$
\Sigma = \mathbb E[(r_t - \mu)(r_t - \mu)^\top]
$$

where:

$$
\mu = \mathbb E[r_t]
$$

For a portfolio with weights $w$, the portfolio return is:

$$
r_{p,t} = w^\top r_t
$$

and the portfolio variance is the quadratic form:

$$
\sigma_p^2 = w^\top \Sigma w
$$

This single expression appears everywhere in quantitative finance: mean-variance optimisation, risk parity, factor models, VaR approximations, minimum variance portfolios, and hedging.

## 2. Synthetic factor-model data

To keep the notebook fully reproducible, we generate synthetic returns using a simple factor model.

Let:

$$
r_t = B f_t + \epsilon_t
$$

where:

- $B$ is the asset-factor loading matrix;
- $f_t$ is a vector of common factor returns;
- $\epsilon_t$ is idiosyncratic asset-specific noise.

The true covariance matrix is:

$$
\begin{aligned}
\Sigma_{\text{true}} &= B \Sigma_f B^\top + D
\end{aligned}
$$

where $D$ is a diagonal matrix of idiosyncratic variances.

This structure is useful because it resembles real markets: assets are partly driven by common risk factors and partly by asset-specific noise.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

In [ ]:
def simulate_factor_returns(
    num_observations: int,
    num_assets: int,
    num_factors: int,
    factor_vol: float = 0.012,
    idiosyncratic_vol_low: float = 0.006,
    idiosyncratic_vol_high: float = 0.018
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """
    Simulate asset returns from a linear factor model.

    r_t = B f_t + epsilon_t

    Parameters
    ----------
    num_observations:
        Number of time observations.

    num_assets:
        Number of assets.

    num_factors:
        Number of latent common factors.

    factor_vol:
        Base volatility scale for the factors.

    idiosyncratic_vol_low:
        Lower bound for idiosyncratic volatility.

    idiosyncratic_vol_high:
        Upper bound for idiosyncratic volatility.

    Returns
    -------
    returns_df:
        DataFrame of synthetic asset returns.

    true_cov:
        True covariance matrix implied by the data generating process.

    loadings:
        Factor loading matrix B.
    """
    # Random but stable factor loadings.
    loadings = rng.normal(loc=0.0, scale=0.7, size=(num_assets, num_factors))

    # Create a positive definite factor covariance matrix.
    raw_factor_matrix = rng.normal(size=(num_factors, num_factors))
    factor_corr = raw_factor_matrix @ raw_factor_matrix.T
    factor_corr = factor_corr / np.sqrt(np.outer(np.diag(factor_corr), np.diag(factor_corr)))

    factor_vols = factor_vol * np.linspace(1.2, 0.6, num_factors)
    factor_cov = np.diag(factor_vols) @ factor_corr @ np.diag(factor_vols)

    # Idiosyncratic variances.
    idiosyncratic_vols = rng.uniform(
        idiosyncratic_vol_low,
        idiosyncratic_vol_high,
        size=num_assets
    )
    idiosyncratic_cov = np.diag(idiosyncratic_vols ** 2)

    true_cov = loadings @ factor_cov @ loadings.T + idiosyncratic_cov

    # Simulate observations.
    factor_returns = rng.multivariate_normal(
        mean=np.zeros(num_factors),
        cov=factor_cov,
        size=num_observations
    )

    noise = rng.normal(
        loc=0.0,
        scale=idiosyncratic_vols,
        size=(num_observations, num_assets)
    )

    returns = factor_returns @ loadings.T + noise

    columns = [f"Asset_{i:02d}" for i in range(1, num_assets + 1)]
    returns_df = pd.DataFrame(returns, columns=columns)

    return returns_df, true_cov, loadings

In [ ]:
num_observations = 750
num_assets = 40
num_factors = 5

returns_df, true_cov, loadings = simulate_factor_returns(
    num_observations=num_observations,
    num_assets=num_assets,
    num_factors=num_factors
)

returns_df.head()

## 3. Estimating the sample covariance matrix

In practice, the true covariance matrix is unknown.

Given observed returns $R$, we estimate covariance using:

$$
\begin{aligned}
\hat{\Sigma} &= \frac{1}{T-1} (X^\top X)
\end{aligned}
$$

where $X$ is the demeaned return matrix.

The challenge is that $\hat{\Sigma}$ can be noisy, ill-conditioned, or even singular when the number of assets is large relative to the number of observations.

In [ ]:
def sample_covariance(returns: pd.DataFrame | np.ndarray) -> np.ndarray:
    """
    Estimate the sample covariance matrix using demeaned returns.
    """
    x = np.asarray(returns)
    x = x - x.mean(axis=0, keepdims=True)
    return (x.T @ x) / (x.shape[0] - 1)


sample_cov = sample_covariance(returns_df)

sample_cov.shape

In [ ]:
def plot_matrix_heatmap(matrix: np.ndarray, title: str) -> None:
    """
    Plot a matrix heatmap using matplotlib.
    """
    plt.figure(figsize=(8, 6))
    plt.imshow(matrix, aspect="auto")
    plt.colorbar(label="Value")
    plt.title(title)
    plt.xlabel("Asset index")
    plt.ylabel("Asset index")
    plt.show()


plot_matrix_heatmap(sample_cov, "Sample Covariance Matrix")

## 4. Positive semi-definiteness and eigenvalues

A valid covariance matrix should be symmetric and positive semi-definite.

Symmetry means:

$$
\Sigma = \Sigma^\top
$$

Positive semi-definiteness means:

$$
x^\top \Sigma x \geq 0
$$

for any vector $x$.

This matters financially because $x^\top \Sigma x$ represents the variance of a portfolio. A variance cannot be negative.

A symmetric matrix is positive semi-definite if all of its eigenvalues are non-negative.

In [ ]:
def covariance_diagnostics(cov: np.ndarray) -> pd.Series:
    """
    Return numerical diagnostics for a covariance matrix.
    """
    eigenvalues = np.linalg.eigvalsh(cov)

    symmetry_error = np.linalg.norm(cov - cov.T, ord="fro")
    min_eigenvalue = eigenvalues.min()
    max_eigenvalue = eigenvalues.max()

    if min_eigenvalue <= 0:
        condition_number = np.inf
    else:
        condition_number = max_eigenvalue / min_eigenvalue

    return pd.Series({
        "symmetry_error_frobenius": symmetry_error,
        "min_eigenvalue": min_eigenvalue,
        "max_eigenvalue": max_eigenvalue,
        "condition_number": condition_number,
        "numerical_rank": np.linalg.matrix_rank(cov)
    })


diagnostics_df = pd.DataFrame({
    "true_cov": covariance_diagnostics(true_cov),
    "sample_cov": covariance_diagnostics(sample_cov)
})

diagnostics_df

In [ ]:
sample_eigenvalues = np.linalg.eigvalsh(sample_cov)
true_eigenvalues = np.linalg.eigvalsh(true_cov)

plt.figure(figsize=(10, 6))
plt.plot(np.sort(true_eigenvalues)[::-1], marker="o", label="True covariance eigenvalues")
plt.plot(np.sort(sample_eigenvalues)[::-1], marker="o", label="Sample covariance eigenvalues")
plt.yscale("log")
plt.title("Eigenvalue Spectrum")
plt.xlabel("Eigenvalue rank")
plt.ylabel("Eigenvalue")
plt.legend()
plt.show()

## 5. Cholesky factorisation and correlated simulations

If a covariance matrix $\Sigma$ is positive definite, we can decompose it as:

$$
\Sigma = L L^\top
$$

where $L$ is lower triangular.

If:

$$
z \sim \mathcal N(0, I)
$$

then:

$$
Lz \sim \mathcal N(0, \Sigma)
$$

This is one of the standard ways to simulate correlated asset returns for Monte Carlo pricing, portfolio risk, and stress testing.

In [ ]:
def cholesky_with_jitter(cov: np.ndarray, jitter: float = 1e-12) -> np.ndarray:
    """
    Compute the Cholesky factor of a covariance matrix.
    Adds small diagonal jitter if needed for numerical stability.
    """
    try:
        return np.linalg.cholesky(cov)
    except np.linalg.LinAlgError:
        adjusted_cov = cov + jitter * np.eye(cov.shape[0])
        return np.linalg.cholesky(adjusted_cov)


chol = cholesky_with_jitter(true_cov)

num_simulated = 100_000
z = rng.standard_normal(size=(num_assets, num_simulated))
simulated_returns = (chol @ z).T

simulated_cov = sample_covariance(simulated_returns)

cholesky_validation = pd.Series({
    "frobenius_error_simulated_vs_true": np.linalg.norm(simulated_cov - true_cov, ord="fro"),
    "relative_frobenius_error": np.linalg.norm(simulated_cov - true_cov, ord="fro") / np.linalg.norm(true_cov, ord="fro")
})

cholesky_validation

## 6. PCA as a risk-factor decomposition

For a symmetric covariance matrix, eigenvalue decomposition gives:

$$
\Sigma = Q \Lambda Q^\top
$$

where:

- $Q$ contains orthonormal eigenvectors;
- $\Lambda$ is a diagonal matrix of eigenvalues.

In finance, the leading eigenvectors often represent dominant risk directions.

For equity returns, the first principal component often behaves like a broad market factor. Later components may represent sector, style, or relative-value structures.

PCA is useful for:

1. reducing dimensionality;
2. detecting hidden correlation;
3. building factor models;
4. stress testing portfolios along major risk directions;
5. denoising covariance matrices.

In [ ]:
def pca_from_covariance(cov: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Perform eigenvalue decomposition of a covariance matrix and sort components by descending eigenvalue.
    """
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = np.argsort(eigenvalues)[::-1]

    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    explained_variance_ratio = eigenvalues / eigenvalues.sum()

    return eigenvalues, eigenvectors, explained_variance_ratio


eigenvalues, eigenvectors, explained_variance_ratio = pca_from_covariance(sample_cov)

pca_summary = pd.DataFrame({
    "component": np.arange(1, num_assets + 1),
    "eigenvalue": eigenvalues,
    "explained_variance_ratio": explained_variance_ratio,
    "cumulative_explained_variance": np.cumsum(explained_variance_ratio)
})

pca_summary.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(
    pca_summary["component"],
    pca_summary["cumulative_explained_variance"],
    marker="o"
)
plt.axhline(0.80, linestyle="--", label="80% variance threshold")
plt.axhline(0.90, linestyle="--", label="90% variance threshold")
plt.title("Cumulative Variance Explained by Principal Components")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.legend()
plt.show()

In [ ]:
num_components = 5

low_rank_cov = (
    eigenvectors[:, :num_components]
    @ np.diag(eigenvalues[:num_components])
    @ eigenvectors[:, :num_components].T
)

residual_variance = np.diag(sample_cov - low_rank_cov)
residual_variance = np.maximum(residual_variance, 0.0)

factor_plus_diag_cov = low_rank_cov + np.diag(residual_variance)

reconstruction_error = pd.Series({
    "low_rank_relative_error": np.linalg.norm(low_rank_cov - sample_cov, ord="fro") / np.linalg.norm(sample_cov, ord="fro"),
    "factor_plus_diag_relative_error": np.linalg.norm(factor_plus_diag_cov - sample_cov, ord="fro") / np.linalg.norm(sample_cov, ord="fro")
})

reconstruction_error

## 7. Condition numbers and matrix inversion risk

The condition number of a covariance matrix is roughly:

$$
\kappa(\Sigma) = \frac{\lambda_{\max}}{\lambda_{\min}}
$$

where $\lambda_{\max}$ and $\lambda_{\min}$ are the largest and smallest eigenvalues.

A large condition number means the matrix is close to singular. In finance, this is dangerous because portfolio optimisation often requires the inverse covariance matrix:

$$
\Sigma^{-1}
$$

Small estimation errors in $\Sigma$ can lead to large errors in $\Sigma^{-1}$, causing unstable and extreme portfolio weights.

In [ ]:
def simulate_covariance_regime(num_observations: int, num_assets: int, num_factors: int) -> pd.Series:
    """
    Simulate returns and return covariance diagnostics for a given T/N regime.
    """
    returns, true_cov_regime, _ = simulate_factor_returns(
        num_observations=num_observations,
        num_assets=num_assets,
        num_factors=num_factors
    )

    cov = sample_covariance(returns)
    diag = covariance_diagnostics(cov)

    diag["num_observations"] = num_observations
    diag["num_assets"] = num_assets
    diag["T_over_N"] = num_observations / num_assets

    return diag


regime_diagnostics = pd.DataFrame([
    simulate_covariance_regime(num_observations=60, num_assets=80, num_factors=5),
    simulate_covariance_regime(num_observations=120, num_assets=80, num_factors=5),
    simulate_covariance_regime(num_observations=750, num_assets=80, num_factors=5),
    simulate_covariance_regime(num_observations=2000, num_assets=80, num_factors=5)
])

regime_diagnostics[
    ["num_observations", "num_assets", "T_over_N", "numerical_rank", "min_eigenvalue", "condition_number"]
]

When $T < N$, the sample covariance matrix is rank-deficient and cannot be safely inverted.

This is one reason high-dimensional portfolio optimisation is difficult: the number of assets can be large while the number of relevant observations is limited.

## 8. Minimum variance portfolio and instability

The global minimum variance portfolio solves:

$$
\min_w w^\top \Sigma w
$$

subject to:

$$
\mathbf{1}^\top w = 1
$$

The unconstrained closed-form solution is:

$$
\begin{aligned}
w^\star &= \frac{\Sigma^{-1}\mathbf{1}} {\mathbf{1}^\top \Sigma^{-1}\mathbf{1}}
\end{aligned}
$$

This formula is elegant, but dangerous in practice because it depends directly on $\Sigma^{-1}$.

If the covariance estimate is noisy or ill-conditioned, the weights can become unstable.

In [ ]:
def global_minimum_variance_weights(cov: np.ndarray, use_pinv: bool = True) -> np.ndarray:
    """
    Compute global minimum variance portfolio weights.

    Uses the pseudo-inverse by default for numerical stability.
    """
    n = cov.shape[0]
    ones = np.ones(n)

    if use_pinv:
        precision = np.linalg.pinv(cov)
    else:
        precision = np.linalg.inv(cov)

    raw_weights = precision @ ones
    weights = raw_weights / (ones @ raw_weights)

    return weights


def portfolio_variance(weights: np.ndarray, cov: np.ndarray) -> float:
    """
    Compute portfolio variance.
    """
    return float(weights.T @ cov @ weights)


sample_weights = global_minimum_variance_weights(sample_cov)
true_weights = global_minimum_variance_weights(true_cov)

weight_summary = pd.DataFrame({
    "sample_cov_weights": sample_weights,
    "true_cov_weights": true_weights
})

weight_summary.describe()

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(np.arange(num_assets), sample_weights, alpha=0.7, label="Weights from sample covariance")
plt.bar(np.arange(num_assets), true_weights, alpha=0.5, label="Weights from true covariance")
plt.axhline(0, linewidth=1)
plt.title("Global Minimum Variance Weights: Sample Covariance vs True Covariance")
plt.xlabel("Asset index")
plt.ylabel("Weight")
plt.legend()
plt.show()

In [ ]:
gmv_risk_comparison = pd.Series({
    "true_cov_weights_true_risk": portfolio_variance(true_weights, true_cov),
    "sample_cov_weights_true_risk": portfolio_variance(sample_weights, true_cov),
    "sample_cov_weights_sample_risk": portfolio_variance(sample_weights, sample_cov)
})

gmv_risk_comparison

The sample-covariance portfolio may look good in-sample, but its true risk can be worse than expected.

This is the central practical problem:

> Optimisation amplifies estimation error.

The optimiser finds the directions that look best under the estimated matrix, but those directions may simply be noise.

## 9. Simple covariance shrinkage

A common stabilisation technique is covariance shrinkage.

Instead of using the raw sample covariance matrix:

$$
\hat{\Sigma}
$$

we blend it with a structured target matrix $F$:

$$
\begin{aligned}
\hat{\Sigma}_{\text{shrunk}} &= (1-\alpha)\hat{\Sigma} + \alpha F
\end{aligned}
$$

where:

$$
0 \leq \alpha \leq 1
$$

In this notebook, we use a simple diagonal target:

$$
F = \operatorname{diag}(\hat{\Sigma})
$$

This is not the full Ledoit-Wolf estimator, but it demonstrates the key idea: shrink noisy off-diagonal correlations towards a more stable structure.

In [ ]:
def shrink_covariance_to_diagonal(sample_cov: np.ndarray, alpha: float) -> np.ndarray:
    """
    Shrink the sample covariance matrix towards its diagonal.

    alpha = 0 gives the sample covariance.
    alpha = 1 gives the diagonal covariance.
    """
    if not 0 <= alpha <= 1:
        raise ValueError("alpha must be between 0 and 1.")

    target = np.diag(np.diag(sample_cov))
    return (1 - alpha) * sample_cov + alpha * target


shrinkage_rows = []

for alpha in np.linspace(0, 1, 21):
    shrunk_cov = shrink_covariance_to_diagonal(sample_cov, alpha=alpha)
    shrunk_weights = global_minimum_variance_weights(shrunk_cov)

    shrinkage_rows.append({
        "alpha": alpha,
        "condition_number": covariance_diagnostics(shrunk_cov)["condition_number"],
        "true_portfolio_variance": portfolio_variance(shrunk_weights, true_cov),
        "max_abs_weight": np.max(np.abs(shrunk_weights)),
        "gross_leverage": np.sum(np.abs(shrunk_weights)),
        "covariance_relative_error": np.linalg.norm(shrunk_cov - true_cov, ord="fro") / np.linalg.norm(true_cov, ord="fro")
    })

shrinkage_df = pd.DataFrame(shrinkage_rows)
shrinkage_df.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(shrinkage_df["alpha"], shrinkage_df["condition_number"], marker="o")
plt.yscale("log")
plt.title("Shrinkage Reduces the Covariance Condition Number")
plt.xlabel(r"Shrinkage intensity $\alpha$")
plt.ylabel("Condition number")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(shrinkage_df["alpha"], shrinkage_df["true_portfolio_variance"], marker="o", label="True portfolio variance")
plt.title("Out-of-Sample Risk Under Shrinkage")
plt.xlabel(r"Shrinkage intensity $\alpha$")
plt.ylabel("Portfolio variance under true covariance")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(shrinkage_df["alpha"], shrinkage_df["gross_leverage"], marker="o", label="Gross leverage")
plt.plot(shrinkage_df["alpha"], shrinkage_df["max_abs_weight"], marker="o", label="Max absolute weight")
plt.title("Shrinkage Stabilises Portfolio Weights")
plt.xlabel(r"Shrinkage intensity $\alpha$")
plt.ylabel("Weight diagnostic")
plt.legend()
plt.show()

## 10. Numerical linear algebra checklist for finance

This notebook motivates a practical checklist.

Before using a covariance matrix in a portfolio, risk, or Monte Carlo model, check:

1. **Symmetry**  
   Is $\Sigma \approx \Sigma^\top$?

2. **Positive semi-definiteness**  
   Are all eigenvalues non-negative up to numerical tolerance?

3. **Condition number**  
   Is the matrix close to singular?

4. **Rank**  
   Is the matrix full rank, especially when $T < N$?

5. **Stability of inverse-based weights**  
   Do small changes in data produce large changes in weights?

6. **Out-of-sample performance**  
   Does the covariance estimate reduce realised risk, or only in-sample risk?

7. **Economic interpretability**  
   Do eigenvectors or principal components correspond to meaningful risk factors?

## 11. Limitations

This notebook is deliberately simplified.

### 11.1 Synthetic data is cleaner than market data

The synthetic factor model has a known true covariance matrix. Real markets do not.

In real data, covariance estimates are affected by:

- changing market regimes;
- stale prices;
- missing data;
- asynchronous trading;
- heavy tails;
- jumps;
- corporate actions;
- survivorship bias.

### 11.2 The diagonal shrinkage target is simplistic

Shrinking to the diagonal removes much of the correlation structure.

Real covariance estimators often use more sophisticated targets, such as:

- constant-correlation targets;
- factor-model covariance targets;
- random-matrix eigenvalue cleaning;
- nonlinear shrinkage;
- robust covariance estimators.

### 11.3 PCA components are not always stable

Principal components can change through time.

A component that appears meaningful in one sample window may rotate or disappear in another.

This matters because portfolio strategies that rely on historical eigenvectors may become unstable when correlations shift.

### 11.4 Minimum variance optimisation ignores expected returns

The global minimum variance portfolio depends only on covariance.

Full mean-variance optimisation also requires expected returns, which are even harder to estimate reliably than covariance.

### 11.5 Inversion-based formulas are fragile

Closed-form formulas involving $\Sigma^{-1}$ are mathematically elegant but numerically dangerous.

In production, constraints, regularisation, robust optimisation, and careful validation are often more important than the unconstrained closed-form solution.

## 12. Research frontier and current directions

Numerical linear algebra remains a live research area in quantitative finance because modern portfolios are high-dimensional, noisy, and non-stationary.

### 12.1 Nonlinear shrinkage and eigenvalue cleaning

The sample covariance matrix is often unreliable when the number of assets is large relative to the number of observations.

Nonlinear shrinkage methods improve covariance estimation by transforming noisy sample eigenvalues while preserving the eigenvectors.

This connects directly to random matrix theory and high-dimensional statistics.

A future notebook could compare:

- sample covariance;
- diagonal shrinkage;
- Ledoit-Wolf linear shrinkage;
- nonlinear eigenvalue shrinkage;
- random matrix theory eigenvalue clipping.

### 12.2 Task-specific covariance cleaning

A covariance matrix that is statistically accurate under Frobenius error is not necessarily optimal for portfolio construction.

Recent research emphasises that covariance estimators should sometimes be judged by the downstream task, such as realised minimum-variance portfolio risk.

This motivates estimators trained or calibrated directly against portfolio risk, not only against matrix reconstruction error.

### 12.3 Robust covariance and tail-risk constraints

Covariance only captures second moments. It may not control losses during extreme market conditions.

Modern portfolio models often combine covariance estimation with tail-risk constraints such as Conditional Value-at-Risk.

A future notebook could compare a minimum-variance portfolio against a CVaR-constrained portfolio under simulated crisis scenarios.

### 12.4 Graph-based and network-based portfolio models

Correlation and covariance matrices can be interpreted as networks.

Graph-based methods attempt to learn relationships between assets using network structure rather than treating the covariance matrix as a flat table of pairwise relationships.

A future notebook could build a minimum-spanning-tree or filtered correlation network and compare it with PCA risk factors.

### 12.5 Neural covariance and precision-matrix estimators

Some recent methods use neural networks to learn covariance cleaning or precision-matrix estimation while preserving useful financial structure.

The key challenge is to avoid black-box overfitting and enforce properties such as symmetry, positive semi-definiteness, interpretability, and stable out-of-sample risk.

A future notebook could train a small neural network to shrink eigenvalues on synthetic covariance matrices and test whether the resulting portfolio risk improves out of sample.

### 12.6 Randomised numerical linear algebra

Large covariance matrices and large Monte Carlo problems can be expensive.

Randomised SVD, sketching, and low-rank approximation methods can accelerate PCA, factor modelling, and risk calculations when the asset universe is very large.

A future notebook could benchmark exact PCA against randomised PCA on a large synthetic asset universe.

## 13. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_04_mean_variance_optimization_ledoit_wolf**  
   Applying covariance shrinkage to Markowitz portfolio optimisation.

2. **04_05_pca_spectral_risk_analysis**  
   Using PCA to detect hidden concentration and fake diversification.

3. **04_09_risk_parity_equal_risk_contribution**  
   Using covariance matrices to allocate risk more evenly.

4. **04_11_cvar_convex_optimization**  
   Moving beyond variance to tail-risk-aware optimisation.

5. **05_07_purged_kfold_and_embargo_cv**  
   Validating financial models without leakage.

6. **06_11_cpp_python_bindings_pybind11**  
   Moving performance-critical linear algebra kernels into compiled code.

7. **07_01_multi_asset_cta_research_pipeline**  
   Applying covariance, PCA, and shrinkage to an end-to-end trading research pipeline.

## 14. Summary

This notebook introduced the numerical linear algebra foundations used throughout quantitative finance.

The key objects were:

1. return vectors;
2. covariance matrices;
3. eigenvalues and eigenvectors;
4. Cholesky factorisation;
5. PCA risk decomposition;
6. condition numbers;
7. covariance shrinkage;
8. inverse-covariance portfolio weights.

The main computational lesson is:

> In finance, the problem is rarely just writing down $w^\top \Sigma w$. The hard part is estimating and inverting $\Sigma$ without letting noise dominate the result.

The main financial lesson is:

> Portfolio optimisation amplifies covariance estimation error, so robust numerical linear algebra is not optional. It is part of the model.

## 15. Further reading

### Textbook foundations

- Trefethen, L. N. and Bau, D. *Numerical Linear Algebra.*
- Golub, G. H. and Van Loan, C. F. *Matrix Computations.*
- Luenberger, D. G. *Investment Science.*
- Glasserman, P. *Monte Carlo Methods in Financial Engineering.*

### Quantitative finance and covariance estimation

- Markowitz, H. "Portfolio Selection."
- Ledoit, O. and Wolf, M. "A Well-Conditioned Estimator for Large-Dimensional Covariance Matrices."
- Ledoit, O. and Wolf, M. "Nonlinear Shrinkage Estimation of Large-Dimensional Covariance Matrices."
- Bongiorno, C. and Challet, D. "Non-linear shrinkage of the price return covariance matrix is far from optimal for portfolio optimisation."
- Zhou, Q. "Portfolio Optimization with Robust Covariance and Conditional Value-at-Risk Constraints."
- Korangi, K., Mues, C., and Bravo, C. "Large-scale Time-Varying Portfolio Optimisation using Graph Attention Networks."